In [7]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR, AUX_COLS, ALPHAS
from src.data   import load_labels, load_coal_spectra
from src.submit import pack_submission

In [2]:
label_map, aux_map = load_labels()
coal_type = COAL_TYPES[0]
train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)

In [3]:
from src.model import get_cv_splits

# get the data
n_batches = train_data['n_batches']
y          = train_data['targets']
aux        = train_data['aux']
groups     = train_data['groups']
coal_mean  = float(y.mean())
splits = get_cv_splits(groups, n_batches)

n_batches, y.shape, aux.shape

(11, (140,), (140, 4))

In [4]:
from src.features import build_feature_matrix
# Get the feature matrix
X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True)


In [106]:
from sklearn.ensemble import HistGradientBoostingRegressor
# Define the base model
base_hgb = HistGradientBoostingRegressor(
    max_iter=150,
    learning_rate=0.05,
    min_samples_leaf=20,
    early_stopping=False,  # CRITICAL: Prevents random shot-level validation leakage
    random_state=42
)

In [14]:
# focus on one aux variable for now
col_idx, col_name = 0, AUX_COLS[0]
col_idx, col_name

(0, '全水分')

In [15]:
y_aux = aux[:, col_idx]
y_aux.shape

(140,)

In [20]:
# focus on one split for now
tr_idx, val_idx = splits[0]
tr_idx.shape, val_idx.shape

((113,), (27,))

In [27]:
# Parameters to search through
PARAM_GRID_HGB = {
    'l2_regularization': [0.0, 0.1, 1.0, 10.0, 50.0],
    'max_depth': [4, 6]
}

In [31]:
from sklearn.model_selection import GridSearchCV, GroupKFold
inner_cv = GroupKFold(n_splits=3)
grid = GridSearchCV(
    estimator=base_hgb,
    param_grid=PARAM_GRID_HGB,
    cv=inner_cv,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

In [32]:
# Fit search using group IDs (pass groups[tr_idx] to maintain group isolation)
grid.fit(X_spec[tr_idx], y_aux[tr_idx], groups=groups[tr_idx])

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",HistGradientB...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'l2_regularization': [0.0, 0.1, ...], 'max_depth': [4, 6]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_root_mean_squared_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",GroupKFold(n_...shuffle=False)
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose:

In [33]:
# Write a full loop over the outer cv folds
oof = np.zeros_like(y_aux, dtype=np.float32)
for tr_idx, val_idx in splits:
    base_hgb = HistGradientBoostingRegressor(
        max_iter=150,
        learning_rate=0.05,
        min_samples_leaf=20,
        early_stopping=False,  # CRITICAL: Prevents random shot-level validation leakage
        random_state=42
    )

    inner_cv = GroupKFold(n_splits=3)
    grid = GridSearchCV(
        estimator=base_hgb,
        param_grid=PARAM_GRID_HGB,
        cv=inner_cv,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )

    grid.fit(X_spec[tr_idx], y_aux[tr_idx], groups=groups[tr_idx])
    oof[val_idx] = grid.best_estimator_.predict(X_spec[val_idx])

In [41]:
np.sqrt(((y_aux - oof) ** 2).mean()).item()

1.3547359921102264

## Integrate into the training pipeline

In [66]:
# Helper fucntions
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, Ridge
from config import ALPHAS, AUX_COLS, SMALL_BATCH_THRESHOLD

def combine_features(X, predicted_aux, interaction_terms = False):
    """
    Combine the original feature matrix with the Step 1 predictions, with the option
    of including interaction terms.

    Input:
        X: The feature matrix
        predicted_aux_oof: The Step 1 prediciton

    Returns:
        The combined feature for Step 2

    """
    if interaction_terms:
        # Additional interaction terms to be considered
        # Moisture = 0, Ash = 1, H = 2, S = 3
        comb_density = 100 - predicted_aux[:, 0] - predicted_aux[:, 1] # Combustion density proxy
        ash_to_moisture = predicted_aux[:, 1] / (predicted_aux[:, 0] + 1e-8) # Ash to Moisture ratio
        hydro_to_ash = predicted_aux[:, 2] / (100 - predicted_aux[:, 1]) # H to Ash displacement
        return np.column_stack([X, predicted_aux, comb_density, ash_to_moisture, hydro_to_ash])
    
    return np.hstack([X, predicted_aux])

def find_best_shrinkage(oof_preds, oof_true, coal_mean):
    """
    在 [0, 1] 上网格搜索混合权重 w，最小化 OOF-RMSE。
    w=1.0 代表纯模型，w=0.0 代表纯均值。
    """
    best_w, best_rmse = 1.0, float('inf')
    for w in np.linspace(0.0, 1.0, 21):
        blended = w * np.array(oof_preds) + (1 - w) * coal_mean
        rmse    = float(np.sqrt(np.mean((blended - np.array(oof_true)) ** 2)))
        if rmse < best_rmse:
            best_rmse, best_w = rmse, w
    return best_w, best_rmse

In [ ]:
def train_coal_model(coal_type, train_data, als_param=(1e6, 0.005), param_search=False):
    """
    训练某煤种的两阶段模型，返回预测所需的全部参数。

    输出 dict 包含:
        spec_scalers  : (scaler_spec, pca, scaler_hand)
        aux_models    : {辅助指标名: RidgeCV 或 None}
        scaler_s2     : Stage2 的特征标准化器
        final_model   : Stage2 最终 RidgeCV
        coal_mean     : 训练集发热量均值（收缩锚点）
        shrink_w      : 收缩权重 w（1.0 = 不收缩）
        cv_rmse       : 交叉验证 RMSE
    """
    
    n_batches  = train_data['n_batches']

    # 光谱 → 特征矩阵（训练集 fit）
    # This function filters out data with anomolous spectral data
    X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True, als_param=als_param)

    y          = train_data['targets']
    aux        = train_data['aux']
    groups     = train_data['groups']
    coal_mean  = float(y.mean())

    if not param_search:
        print(f"\n  [{coal_type}]  {n_batches}批次  {len(y)}条光谱  "
            f"Q={y.min():.0f}~{y.max():.0f}")

    splits = get_cv_splits(groups, n_batches)

    # ── Stage 1: 光谱 → 辅助指标 (OOF) ────────────────────────────────────
    aux_models        = {}
    predicted_aux_oof = np.zeros_like(aux, dtype=np.float32)

    # 1. Define hyperparameter search space for Stage 1 tree models
    PARAM_GRID_HGB = {
        'l2_regularization': [0.0, 1e-3, 1e-2, 0.1, 1.0, 10.0, 100.0],
        'max_depth': [3, 4, 5],
        'max_iter': np.linspace(300, 800, 6, dtype=int).tolist(),

    }

    for col_idx, col_name in enumerate(AUX_COLS):
        y_aux = aux[:, col_idx]

        # 辅助指标有缺失时退化为用均值填充
        if np.isnan(y_aux).any():
            predicted_aux_oof[:, col_idx] = float(np.nanmean(y_aux))
            aux_models[col_name] = None
            continue

        oof = np.zeros(len(y_aux), dtype=np.float32)

        # --- GLOBAL PARAMETER DISCOVERY ---
        # If batch size is too small, fallback to robust default parameters
        # instead of trusting GridSearch on tiny folds.
        if n_batches <= SMALL_BATCH_THRESHOLD:
            best_params = {'l2_regularization': 10.0, 'max_depth': 3}
            if not param_search:
                print(f"      [{col_name}] Small batch detected. Using robust fallback params.")
        else:
            # Do ONE global GridSearch using the outer splits to find a stable configuration
            base_hgb_search = HistGradientBoostingRegressor(
                learning_rate=0.05, 
                min_samples_leaf=20, 
                early_stopping=False, 
                random_state=42
            )
            # Use outer splits directly for the search to evaluate on hold-out groups
            grid = GridSearchCV(
                estimator=base_hgb_search, param_grid=PARAM_GRID_HGB,
                cv=splits, scoring='neg_root_mean_squared_error', n_jobs=-1
            )
            grid.fit(X_spec, y_aux)
            best_params = grid.best_params_

        # --- CONSISTENT OOF GENERATION ---
        # Now use the EXACT SAME parameters across all folds to prevent Frankenstein OOFs
        for tr_idx, val_idx in splits:
            fold_model = HistGradientBoostingRegressor(
                learning_rate=0.05,
                min_samples_leaf=20,
                early_stopping=False,
                random_state=42,
                **best_params  # Inject the globally best parameters
            )
            
            fold_model.fit(X_spec[tr_idx], y_aux[tr_idx])
            oof[val_idx] = fold_model.predict(X_spec[val_idx])

        predicted_aux_oof[:, col_idx] = oof

        # 3. 全量重新拟合 (Tune & Fit on full dataset for downstream inference model)
        final_aux_model = HistGradientBoostingRegressor(
            learning_rate=0.05,
            min_samples_leaf=20,
            early_stopping=False,
            random_state=42,
            **best_params
        )
        final_aux_model.fit(X_spec, y_aux)
        
        # Store the best full model for inference
        aux_models[col_name] = final_aux_model

    # ── Stage 2: [光谱特征 + 预测辅助指标] → 发热量 ──────────────────────
    
    X_s2      = combine_features(X_spec, predicted_aux_oof)
    scaler_s2 = StandardScaler()
    X_s2      = scaler_s2.fit_transform(np.nan_to_num(X_s2))

    # OOF 批次预测（用于计算 CV-RMSE 和收缩权重）
    oof_batch_preds, oof_batch_true, batch_rmses = [], [], []

    for tr_idx, val_idx in splits:
        m2 = RidgeCV(alphas=ALPHAS)
        m2.fit(X_s2[tr_idx], y[tr_idx])
        val_pred   = m2.predict(X_s2[val_idx])
        val_groups = groups[val_idx]

        fold_se = []
        for bg in np.unique(val_groups):
            mask   = val_groups == bg
            true_q = float(y[val_idx][mask][0])
            pred_q = float(np.median(val_pred[mask]))
            oof_batch_preds.append(pred_q)
            oof_batch_true.append(true_q)
            fold_se.append((true_q - pred_q) ** 2)
        batch_rmses.append(float(np.sqrt(np.mean(fold_se))))

    cv_rmse_raw = float(np.mean(batch_rmses))

    # 小批次煤种: 搜索最优收缩权重
    best_w = 1.0
    if n_batches <= SMALL_BATCH_THRESHOLD:
        best_w, cv_rmse_shrunk = find_best_shrinkage(
            oof_batch_preds, oof_batch_true, coal_mean)
        cv_rmse = cv_rmse_shrunk
        if not param_search:
            print(f"    Stage2 CV-RMSE: raw={cv_rmse_raw:.2f}  "
                f"shrunk={cv_rmse_shrunk:.2f}  w={best_w:.2f}")
    else:
        cv_rmse = cv_rmse_raw
        if not param_search:
            print(f"    Stage2 CV-RMSE: {cv_rmse:.2f} ± {np.std(batch_rmses):.2f}")

    # 全量数据重新拟合最终模型
    final_model = RidgeCV(alphas=ALPHAS)
    final_model.fit(X_s2, y)
    if not param_search:
        print(f"    最优正则化 alpha: {final_model.alpha_:.1f}")

    return {
        'spec_scalers': (scaler_spec, pca, scaler_hand),
        'aux_models':   aux_models,
        'scaler_s2':    scaler_s2,
        'final_model':  final_model,
        'coal_mean':    coal_mean,
        'shrink_w':     best_w,
        'cv_rmse':      cv_rmse,
    }

In [70]:
cv_results = {}
for coal_type in COAL_TYPES:
    train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)
    if train_data is None or train_data['n_batches'] == 0:
        print(f"\n  [{coal_type}] 未找到训练数据，跳过")
        continue

    model_dict = train_coal_model(coal_type, train_data)
    cv_results[coal_type] = model_dict['cv_rmse']

global_cv_rmse = float(np.mean(list(cv_results.values())))
print(f"\n{'=' * 60}")
print(f"全局 CV-RMSE: {global_cv_rmse:.2f}")


  [赵固一矿豫焦末煤]  11批次  140条光谱  Q=4708~6143
    Stage2 CV-RMSE: 261.39 ± 173.37
    最优正则化 alpha: 1.0

  [赵固二矿中煤矿]  7批次  86条光谱  Q=3964~4455
      [全水分] Small batch detected. Using robust fallback params.
      [灰分] Small batch detected. Using robust fallback params.
      [氢] Small batch detected. Using robust fallback params.
      [硫] Small batch detected. Using robust fallback params.
    Stage2 CV-RMSE: raw=115.70  shrunk=142.07  w=1.00
    最优正则化 alpha: 10.0

  [中马矿中煤矿]  7批次  96条光谱  Q=2752~3848
      [全水分] Small batch detected. Using robust fallback params.
      [灰分] Small batch detected. Using robust fallback params.
      [氢] Small batch detected. Using robust fallback params.
      [硫] Small batch detected. Using robust fallback params.
    Stage2 CV-RMSE: raw=262.43  shrunk=314.97  w=1.00
    最优正则化 alpha: 50.0

  [九里山矿中煤矿]  18批次  262条光谱  Q=3379~4174
    Stage2 CV-RMSE: 179.93 ± 70.76
    最优正则化 alpha: 1.0

  [煤场混煤]  27批次  360条光谱  Q=3555~4787
    Stage2 CV-RMSE: 295.37 ± 91.52
    最

### Global parameter search for linear model

In [ ]:
def train_coal_model(coal_type, train_data, als_param=(1e6, 0.005), param_search=False):
    """
    训练某煤种的两阶段模型，返回预测所需的全部参数。

    输出 dict 包含:
        spec_scalers  : (scaler_spec, pca, scaler_hand)
        aux_models    : {辅助指标名: RidgeCV 或 None}
        scaler_s2     : Stage2 的特征标准化器
        final_model   : Stage2 最终 RidgeCV
        coal_mean     : 训练集发热量均值（收缩锚点）
        shrink_w      : 收缩权重 w（1.0 = 不收缩）
        cv_rmse       : 交叉验证 RMSE
    """
    
    n_batches  = train_data['n_batches']

    # 光谱 → 特征矩阵（训练集 fit）
    # This function filters out data with anomolous spectral data
    X_spec, scaler_spec, pca, scaler_hand = build_feature_matrix(
        train_data, n_batches, fit=True, als_param=als_param)

    y          = train_data['targets']
    aux        = train_data['aux']
    groups     = train_data['groups']
    coal_mean  = float(y.mean())

    if not param_search:
        print(f"\n  [{coal_type}]  {n_batches}批次  {len(y)}条光谱  "
            f"Q={y.min():.0f}~{y.max():.0f}")

    splits = get_cv_splits(groups, n_batches)

    # ── Stage 1: 光谱 → 辅助指标 (OOF) ────────────────────────────────────
    aux_models        = {}
    predicted_aux_oof = np.zeros_like(aux, dtype=np.float32)

    for col_idx, col_name in enumerate(AUX_COLS):
        y_aux = aux[:, col_idx]

        # 辅助指标有缺失时退化为用均值填充
        if np.isnan(y_aux).any():
            predicted_aux_oof[:, col_idx] = float(np.nanmean(y_aux))
            aux_models[col_name] = None
            continue

        # Global fit for parameter search
        model = RidgeCV(alphas=ALPHAS)
        model.fit(X_spec, y_aux)
        best_alpha = model.alpha_

        m = Ridge(alpha=best_alpha)
        oof = np.zeros(len(y_aux))
        # Foldwise model for intermediate prediction
        for tr_idx, val_idx in splits:
            m.fit(X_spec[tr_idx], y_aux[tr_idx])
            oof[val_idx] = m.predict(X_spec[val_idx])
        predicted_aux_oof[:, col_idx] = oof

        m.fit(X_spec, y_aux)   # 全量重新拟合，存入 aux_models 供推理用
        aux_models[col_name] = m

    # ── Stage 2: [光谱特征 + 预测辅助指标] → 发热量 ──────────────────────
    
    X_s2      = combine_features(X_spec, predicted_aux_oof)
    scaler_s2 = StandardScaler()
    X_s2      = scaler_s2.fit_transform(np.nan_to_num(X_s2))

    # OOF 批次预测（用于计算 CV-RMSE 和收缩权重）
    oof_batch_preds, oof_batch_true, batch_rmses = [], [], []

    for tr_idx, val_idx in splits:
        m2 = RidgeCV(alphas=ALPHAS)
        m2.fit(X_s2[tr_idx], y[tr_idx])
        val_pred   = m2.predict(X_s2[val_idx])
        val_groups = groups[val_idx]

        fold_se = []
        for bg in np.unique(val_groups):
            mask   = val_groups == bg
            true_q = float(y[val_idx][mask][0])
            pred_q = float(np.median(val_pred[mask]))
            oof_batch_preds.append(pred_q)
            oof_batch_true.append(true_q)
            fold_se.append((true_q - pred_q) ** 2)
        batch_rmses.append(float(np.sqrt(np.mean(fold_se))))

    cv_rmse_raw = float(np.mean(batch_rmses))

    # 小批次煤种: 搜索最优收缩权重
    best_w = 1.0
    if n_batches <= SMALL_BATCH_THRESHOLD:
        best_w, cv_rmse_shrunk = find_best_shrinkage(
            oof_batch_preds, oof_batch_true, coal_mean)
        cv_rmse = cv_rmse_shrunk
        if not param_search:
            print(f"    Stage2 CV-RMSE: raw={cv_rmse_raw:.2f}  "
                f"shrunk={cv_rmse_shrunk:.2f}  w={best_w:.2f}")
    else:
        cv_rmse = cv_rmse_raw
        if not param_search:
            print(f"    Stage2 CV-RMSE: {cv_rmse:.2f} ± {np.std(batch_rmses):.2f}")

    # 全量数据重新拟合最终模型
    final_model = RidgeCV(alphas=ALPHAS)
    final_model.fit(X_s2, y)
    if not param_search:
        print(f"    最优正则化 alpha: {final_model.alpha_:.1f}")

    return {
        'spec_scalers': (scaler_spec, pca, scaler_hand),
        'aux_models':   aux_models,
        'scaler_s2':    scaler_s2,
        'final_model':  final_model,
        'coal_mean':    coal_mean,
        'shrink_w':     best_w,
        'cv_rmse':      cv_rmse,
    }

In [68]:
model_dict = train_coal_model(coal_type, train_data)


  [赵固一矿豫焦末煤]  11批次  140条光谱  Q=4708~6143
    Stage2 CV-RMSE: 183.31 ± 88.74
    最优正则化 alpha: 1.0


In [53]:
np.linspace(300, 800, 6, dtype=int).tolist()

[300, 400, 500, 600, 700, 800]

## PLS2 for predicting auxiliary variables

In [71]:
aux.shape

(140, 4)

In [80]:
from sklearn.cross_decomposition import PLSRegression
pls = PLSRegression(n_components=aux.shape[1])
oof = np.zeros_like(aux)

for tr_idx, val_idx in splits:
    pls.fit(X_spec[tr_idx], aux[tr_idx])
    oof[val_idx] = pls.predict(X_spec[val_idx])

In [81]:
for col_idx, col_name in enumerate(AUX_COLS):
    rmse = np.sqrt(np.mean((oof[:, col_idx] - aux[:, col_idx]) ** 2))
    print(f"Aux variable {col_name}")
    print(f"  RMSE: {rmse: .4f}")
    print(f"  Percentage error: {rmse / np.mean(aux[:, col_idx]) * 100: .2f}%")

Aux variable 全水分
  RMSE:  1.4206
  Percentage error:  15.27%
Aux variable 灰分
  RMSE:  3.7852
  Percentage error:  17.37%
Aux variable 氢
  RMSE:  0.0827
  Percentage error:  3.64%
Aux variable 硫
  RMSE:  0.0591
  Percentage error:  13.47%


In [ ]:
from sklearn.linear_model import LassoCV
# Testing several models on a single aux var
col_idx = 3
col_name = AUX_COLS[col_idx]

y_aux = aux[:, col_idx]
ridge = RidgeCV(alphas = ALPHAS)
pls = PLSRegression(n_components=aux.shape[1])
lasso = LassoCV(alphas = ALPHAS)
oof_ridge = np.zeros(len(y_aux))
oof_pls = np.zeros(len(y_aux))
oof_lasso = np.zeros(len(y_aux))

for tr_idx, val_idx in splits:
    ridge.fit(X_spec[tr_idx], y_aux[tr_idx])
    pls.fit(X_spec[tr_idx], y_aux[tr_idx])
    lasso.fit(X_spec[tr_idx], y_aux[tr_idx])
    oof_ridge[val_idx] = ridge.predict(X_spec[val_idx])
    oof_pls[val_idx] = pls.predict(X_spec[val_idx])
    oof_lasso[val_idx] = lasso.predict(X_spec[val_idx])

print(f"Aux variable: {col_name}")
print("="*30)
print("Model tested: Ridge")
rmse = np.sqrt(np.mean((oof_ridge - y_aux) ** 2))
print(f"  RMSE: {rmse: .4f}")
print(f"  Percentage error: {rmse / np.mean(aux[:, col_idx]) * 100: .2f}%")
print("Model tested: PLS")
rmse = np.sqrt(np.mean((oof_pls - y_aux) ** 2))
print(f"  RMSE: {rmse: .4f}")
print(f"  Percentage error: {rmse / np.mean(aux[:, col_idx]) * 100: .2f}%")
rmse = np.sqrt(np.mean((oof_lasso - y_aux) ** 2))
print("Model tested: Lasso")
print(f"  RMSE: {rmse: .4f}")
print(f"  Percentage error: {rmse / np.mean(aux[:, col_idx]) * 100: .2f}%")

Aux variable 硫
Model tested: Ridge
  RMSE:  0.0602
  Percentage error:  13.72%
Model tested: PLS
  RMSE:  0.0599
  Percentage error:  13.66%
Model tested: Lasso
  RMSE:  0.0597
  Percentage error:  13.60%


In [5]:
# Extend to an automated pipeline
import numpy as np
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

def bench_mark_model(col_idx):
    # 1. Setup Target Variable
    #col_idx = 3  # Assuming 3 is Sulfur
    col_name = AUX_COLS[col_idx]
    y_aux = aux[:, col_idx]

    # 2. Define a dictionary of powerful, diverse models
    # Note: SVR and PLS benefit heavily from pre-scaled features, so we wrap SVR in a pipeline.
    models = {
        "Ridge (Baseline)": RidgeCV(alphas=ALPHAS),

        "Lasso": LassoCV(alphas=ALPHAS),
        
        # GridSearch dynamically finds the optimal latent variables for your specific spectra
        "PLS (Tuned)": GridSearchCV(
            PLSRegression(), 
            param_grid={"n_components": [5, 10, 15, 20, 25, 30]}, 
            cv=3
        ),
        
        # ElasticNet is far more stable on collinear LIBS spectra than pure Lasso
        "ElasticNet": ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9], cv=3, max_iter=5000),
        
        # SVR with RBF kernel is fantastic for non-linear matrix effects
        "SVR (RBF Kernel)": make_pipeline(
            StandardScaler(), 
            SVR(C=10.0, epsilon=0.01)
        ),
        
        # Tree models excel at finding indirect proxies (like Fe lines representing FeS2)
        "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42),
        
        "Gradient Boosting": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, random_state=42)
    }

    # 3. Automated Benchmarking Loop
    results = {}

    print(f"=== Benchmarking Models for: {col_name} ===")
    print(f"Mean Target Value: {np.mean(y_aux):.4f}\n")

    for name, model in models.items():
        oof_preds = np.zeros(len(y_aux))
        
        # Foldwise fitting and out-of-fold prediction
        for tr_idx, val_idx in splits:
            X_tr, y_tr = X_spec[tr_idx], y_aux[tr_idx]
            X_val = X_spec[val_idx]
            
            model.fit(X_tr, y_tr)
            oof_preds[val_idx] = model.predict(X_val)
            
        # Calculate mathematically correct RMSE and Relative Error
        rmse = np.sqrt(np.mean((oof_preds - y_aux) ** 2))
        rel_error = (rmse / np.mean(y_aux)) * 100
        
        results[name] = {"RMSE": rmse, "RelError": rel_error, "OOF": oof_preds}
        
        # Print real-time diagnostic
        print(f"Model: {name:<20} | RMSE: {rmse:.4f} | Relative Error: {rel_error:.2f}%")
        
        # If it's PLS, let's see how many components it actually chose on the last fold
        if name == "PLS (Tuned)":
            print(f"  -> Optimal PLS components chosen: {model.best_params_['n_components']}")

In [8]:
for col_idx in range(4):
    bench_mark_model(col_idx)

=== Benchmarking Models for: 全水分 ===
Mean Target Value: 9.3029

Model: Ridge (Baseline)     | RMSE: 1.4088 | Relative Error: 15.14%
Model: Lasso                | RMSE: 1.2579 | Relative Error: 13.52%
Model: PLS (Tuned)          | RMSE: 1.3860 | Relative Error: 14.90%
  -> Optimal PLS components chosen: 5
Model: ElasticNet           | RMSE: 1.3002 | Relative Error: 13.98%
Model: SVR (RBF Kernel)     | RMSE: 1.5398 | Relative Error: 16.55%
Model: Random Forest        | RMSE: 1.4246 | Relative Error: 15.31%
Model: Gradient Boosting    | RMSE: 1.3783 | Relative Error: 14.82%
=== Benchmarking Models for: 灰分 ===
Mean Target Value: 21.7869

Model: Ridge (Baseline)     | RMSE: 3.7234 | Relative Error: 17.09%
Model: Lasso                | RMSE: 4.3067 | Relative Error: 19.77%
Model: PLS (Tuned)          | RMSE: 3.7179 | Relative Error: 17.06%
  -> Optimal PLS components chosen: 5
Model: ElasticNet           | RMSE: 4.3376 | Relative Error: 19.91%
Model: SVR (RBF Kernel)     | RMSE: 4.0984 | Rel